In [3]:
import pandas as pd
import numpy as np

### binning backscatter + fluorescence data
df_bsp = pd.read_excel("Backscatter_fluor_data.xlsx")

# --------------------------------------------------------------------------------------
# 1. Filtering out descending profiles 
# --------------------------------------------------------------------------------------

asc_desc_profiles = (
    df_bsp.groupby(["WMO", "profile_number"])["profile_direction"]
    .apply(lambda x: {"A", "D"}.issubset(set(x)))
)

asc_desc_profiles = asc_desc_profiles[asc_desc_profiles].index

# Extract rows for those profiles
df_a_d = (
    df_bsp.set_index(["WMO", "profile_number"])
    .loc[asc_desc_profiles]
    .reset_index()
)

df_a_d.to_excel("float_profile_a_vs_d.xlsx", index=False)

# Remove descending rows
df_bsp = df_bsp[df_bsp["profile_direction"] != "D"]


# --------------------------------------------------------------------------------------
# 2. Changing data types
# --------------------------------------------------------------------------------------

# Ensure numeric types
df_bsp["PRES"] = pd.to_numeric(df_bsp["PRES"], errors="coerce")
df_bsp["BBP700_ADJUSTED"] = pd.to_numeric(df_bsp["BBP700_ADJUSTED"], errors="coerce")
df_bsp["CHLA_FLUORESCENCE_ADJUSTED"] = pd.to_numeric(df_bsp["CHLA_FLUORESCENCE_ADJUSTED"], errors="coerce")

# --------------------------------------------------------------------------------------
# 3. QC filtering
# --------------------------------------------------------------------------------------

# Remove BBP700 values where QC = 3 or 4
df_bsp.loc[df_bsp["BBP700_ADJUSTED_QC"].isin([3, 4]), "BBP700_ADJUSTED"] = pd.NA

# Remove CHLA fluorescence values where QC = 3 or 4
df_bsp.loc[df_bsp["CHLA_FLUORESCENCE_ADJUSTED_QC"].isin([3, 4]), "CHLA_FLUORESCENCE_ADJUSTED"] = pd.NA

# ---------------------------------------------------------
# 4. Depth binning
# ---------------------------------------------------------

depth_bins = [
    0,10,20,30,40,50,60,70,80,90,100,120,140,160,180,200,220,240,260,280,300,
    320,340,360,380,400,420,440,460,480,500,550,600,650,700,750,800,850,900,
    950,1000,1050,1100,1150,1200,1250,1300,1350,1400,1450,1500,1550,1600,1650,
    1700,1750,1800,1850,1900,1950,2000
]

# Create labels for bins (e.g., "0–10", "10–20")
depth_labels = [f"{depth_bins[i]}–{depth_bins[i+1]}" for i in range(len(depth_bins)-1)]

df_bsp["depth_bin"] = pd.cut(
    df_bsp["PRES"],
    bins=depth_bins,
    labels=depth_labels,
    include_lowest=True,
    right=False
)

# ---------------------------------------------------------
# 5. Compute mean concentration for each (depth_bin, backscatter/fluorescence) and export to spreadsheet
# ---------------------------------------------------------

# Pivot for BBP700
bbp_binned = pd.pivot_table(
    df_bsp,
    values="BBP700_ADJUSTED",
    index="profile_number",
    columns="depth_bin",
    aggfunc="mean"
)

# Save BBP output
bbp_binned.to_excel("BBP_binned.xlsx")


# Pivot for CHLA fluorescence
chl_binned = pd.pivot_table(
    df_bsp,
    values="CHLA_FLUORESCENCE_ADJUSTED",
    index="profile_number",
    columns="depth_bin",
    aggfunc="mean"
)

# Save CHLA output
chl_binned.to_excel("Chl_fluor_binned.xlsx")




C:\Users\steph\AppData\Local\Temp\ipykernel_12516\2641791938.py:77: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  bbp_binned = pd.pivot_table(
C:\Users\steph\AppData\Local\Temp\ipykernel_12516\2641791938.py:90: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  chl_binned = pd.pivot_table(
